In [2]:
import clip
import scipy
import torch
import numpy as np
from torch import nn
import torch.nn.functional as F
from transformers import pipeline
from skimage.segmentation import slic
from torchmetrics.functional import pairwise_cosine_similarity

In [ ]:
model, preprocess = clip.load('ViT-L/14')

In [24]:
concepts = torch.load('concepts/CUB_200_2011/LCDA/concepts_clip_RN50_normalized.pt').to(torch.float64)

In [49]:
torch.cov(concepts.t()).inverse()

tensor([[ 1.9843e+17, -8.1736e+16,  1.6374e+17,  ...,  1.5775e+15,
          7.5683e+16,  8.2091e+16],
        [ 5.7798e+16,  1.3095e+16,  3.6324e+16,  ..., -3.6781e+16,
          9.4044e+16,  1.1110e+17],
        [-4.6751e+16,  5.2422e+16, -2.0962e+17,  ..., -3.5538e+16,
          3.5129e+16, -1.5977e+17],
        ...,
        [-5.6878e+16, -1.3306e+17, -2.5871e+16,  ...,  2.9054e+17,
          6.3907e+16, -3.8770e+16],
        [ 3.2293e+16,  1.4470e+16,  4.5568e+16,  ..., -2.4782e+16,
          7.0203e+16,  1.9333e+16],
        [-6.6704e+16,  3.3947e+16, -1.4607e+17,  ..., -1.9826e+16,
         -3.9669e+16,  4.2080e+17]], dtype=torch.float64)

In [52]:
np.linalg.inv(np.cov(concepts.t()))

array([[ 4.61673336e+17,  7.50958206e+16,  5.06616363e+17, ...,
         2.24371683e+16,  7.26643584e+16, -1.06547376e+17],
       [-5.43229242e+16,  3.04663626e+17,  1.09234071e+17, ...,
         3.18281873e+17, -7.30779362e+15,  1.15667770e+17],
       [-1.04711197e+17, -3.43493391e+16,  1.23544781e+17, ...,
         1.04554419e+17,  1.62595032e+17, -6.11887490e+15],
       ...,
       [-2.43869010e+17, -1.22741365e+17, -3.93120649e+17, ...,
        -5.83744078e+16,  7.15796318e+16,  9.25485572e+16],
       [-1.38464132e+17, -8.36397906e+16, -5.22994084e+17, ...,
        -2.70776564e+17,  2.91692051e+17,  2.59149769e+16],
       [-1.55274787e+17, -1.13409118e+17, -4.79661664e+17, ...,
        -2.53202554e+17, -2.14942908e+17,  2.05308951e+17]])

In [25]:
inv_cov = torch.cov(concepts.t()).inverse()

In [57]:
def gaussian_kernel(x, y, sigma=1.0):
    x_size = x.size(0)
    y_size = y.size(0)
    dim = x.size(1)
    xx = torch.bmm(x, x.transpose(1, 2)).reshape(x_size, dim, -1)
    yy = torch.bmm(y, y.transpose(1, 2)).reshape(y_size, dim, -1)
    zz = torch.bmm(x, y.transpose(1, 2)).reshape(x_size, dim, -1)
    xx = torch.sum(xx * xx, dim=1)
    yy = torch.sum(yy * yy, dim=1)
    zz = torch.sum(zz * zz, dim=1)
    return torch.exp(-0.5 * ((xx.unsqueeze(1) + yy.unsqueeze(0) - 2 * zz) / (sigma ** 2)))

def mmd(x, y, kernel=gaussian_kernel):
    x_size = x.size(0)
    y_size = y.size(0)
    kx = kernel(x, x)
    ky = kernel(y, y)
    kxy = kernel(x, y)
    mx = torch.mean(kx, dim=0)
    my = torch.mean(ky, dim=0)
    d = torch.mean(kx) + torch.mean(ky) - 2 * torch.mean(kxy)
    return d

x = torch.randn(1, 100, 10)
y = torch.randn(1, 100, 10) + 1.0

mmd_value = mmd(x, y)
print("MMD value:", mmd_value.item())

MMD value: -inf


In [58]:
def mmd(x, y, sigma):
    # Implementation from https://torchdrift.org/notebooks/note_on_mmd.html
    # compare kernel MMD paper and code:
    # A. Gretton et al.: A kernel two-sample test, JMLR 13 (2012)
    # http://www.gatsby.ucl.ac.uk/~gretton/mmd/mmd.htm
    # x shape [n, d] y shape [m, d]
    # n_perm number of bootstrap permutations to get p-value, pass none to not get p-value
    n, d = x.shape
    m, d2 = y.shape
    assert d == d2
    xy = torch.cat([x.detach(), y.detach()], dim=0)
    dists = torch.cdist(xy, xy, p=2.0)
    # we are a bit sloppy here as we just keep the diagonal and everything twice
    # note that sigma should be squared in the RBF to match the Gretton et al heuristic
    k = torch.exp((-1/(2*sigma**2)) * dists**2) + torch.eye(n+m)*1e-5
    k_x = k[:n, :n]
    k_y = k[n:, n:]
    k_xy = k[:n, n:]
    # The diagonals are always 1 (up to numerical error, this is (3) in Gretton et al.)
    # note that their code uses the biased (and differently scaled mmd)
    mmd = k_x.sum() / (n * (n - 1)) + k_y.sum() / (m * (m - 1)) - 2 * k_xy.sum() / (n * m)
    return mmd

In [75]:
x = torch.randn(100, 1024)
y = torch.randn(1000, 1024)

dists = torch.pdist(torch.cat([x, y], dim=0))
sigma = dists.median()/2
sigma

tensor(22.6557)

In [76]:
mmd(x, y, sigma)

tensor(0.0112)

In [86]:
# x = torch.randn(100, 1024)
x = torch.empty(100, 1024).normal_(mean=4,std=100)
y = concepts

dists = torch.pdist(torch.cat([x, y], dim=0))
sigma = dists.median()/2
mmd(x, y, sigma)

tensor(0.1892, dtype=torch.float64)

In [90]:
x = concepts[:50, :]
y = concepts

dists = torch.pdist(torch.cat([x, y], dim=0))
sigma = dists.median()/2
mmd(x, y, sigma)

tensor(0.0164, dtype=torch.float64)

In [83]:
x

tensor([[-0.0040, -0.0082, -0.0045,  ..., -0.0207, -0.0011,  0.0035],
        [ 0.0067,  0.0062, -0.0156,  ...,  0.0083,  0.0149,  0.0077],
        [-0.0085, -0.0022, -0.0218,  ...,  0.0008, -0.0087, -0.0072],
        ...,
        [ 0.0047, -0.0011, -0.0042,  ...,  0.0007,  0.0097, -0.0098],
        [-0.0137,  0.0035,  0.0034,  ...,  0.0009,  0.0099, -0.0147],
        [-0.0057,  0.0099, -0.0095,  ..., -0.0009,  0.0142,  0.0055]],
       dtype=torch.float64)

In [92]:
x.shape == y.shape

False